#### Bioassay MoA Master tb

In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
from pathlib import Path
import pandas as pd
import numpy as np
import os

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


def map_assay_layer(assay_type):
    """Maps specific assay types to the defined biological layer hierarchy."""
    if pd.isna(assay_type): 
        return 'Unknown'
    
    assay_type = str(assay_type).strip()
    if assay_type == 'binding_only': 
        return 'Layer 1 (Binding)'
    elif assay_type in ['cAMP', 'Ca2+', 'GTPgS', 'IP1']: 
        return 'Layer 2 (Proximal)'
    elif assay_type == 'beta-arrestin': 
        return 'Layer 3 (Biased)'
    elif assay_type == 'reporter_other': 
        return 'Layer 4 (Reporter)'
    
    return 'Unknown'

def standardize_moa(moa):
    """Standardizes MoA terminologies across different databases."""
    if pd.isna(moa): 
        return 'Unknown'
    
    moa = str(moa).strip().lower()
    if moa in ['full agonist', 'agonist']: return 'Agonist'
    elif moa in ['antagonist']: return 'Antagonist'
    elif moa in ['partial agonist']: return 'Partial Agonist'
    elif moa in ['inverse agonist']: return 'Inverse Agonist'
    elif moa in ['pam']: return 'PAM'
    elif moa in ['nam']: return 'NAM'
    elif moa in ['binder_unknown_moa', 'binder', 'binding_only']: return 'Binder'
    elif moa in ['non-binder']: return 'Non-binder'
    elif moa in ['inactive']: return 'Inactive'
    
    return 'Unknown'

In [ ]:
# ==========================================
# 1. ChEMBL & BindingDB
# ==========================================
print("Processing ChEMBL & BindingDB...")
chembl_bdb = pd.read_csv(str(BASE / "Output/DB/GPCRactDB/ChEMBL_BDB_Assay_Labeled.csv"))
cb_df = chembl_bdb[['Ligand_InChIKey', 'GPCR_UniProt', 'Source_DB', 'Assay_Type_LLM', 'Target_MoA_LLM', 'Final_MoA']].copy()

# Map the layers
cb_df['Assay_Layer'] = cb_df['Assay_Type_LLM'].apply(map_assay_layer)

# Reconstruct MoA: If Final_MoA specifies Binder/Inactive/Non-binder, respect it. Otherwise, use the true MoA label.
def get_cb_moa(row):
    fm = str(row['Final_MoA'])
    if fm in ['Binder_Unknown_MoA', 'Non-binder', 'Inactive', 'Unknown']:
        return fm
    return row['Target_MoA_LLM']

cb_df['MoA'] = cb_df.apply(get_cb_moa, axis=1).apply(standardize_moa)
cb_df = cb_df[['Ligand_InChIKey', 'GPCR_UniProt', 'Assay_Layer', 'MoA', 'Source_DB']]

In [ ]:
# ==========================================
# 2. PubChem
# ==========================================
print("Processing PubChem...")
pubchem = pd.read_csv(str(BASE / "Output/DB/PubChem/NAR/GPCRactDB_v2_PubChem_Quantitative_Finalized.csv"))
pc_df = pubchem[['InChIKey', 'UniProt_AC', 'Assay_Type_LLM', 'Target_MoA_LLM', 'Activity_Outcome']].copy()

# Rename columns to match master schema
pc_df.rename(columns={'InChIKey': 'Ligand_InChIKey', 'UniProt_AC': 'GPCR_UniProt'}, inplace=True)
pc_df['Source_DB'] = 'PubChem'
pc_df['Assay_Layer'] = pc_df['Assay_Type_LLM'].apply(map_assay_layer)

def get_pc_moa(row):
    # Activity_Outcome == 1 means inactive in PubChem
    if row['Activity_Outcome'] == 1:
        return 'Inactive'
    if row['Assay_Type_LLM'] == 'binding_only':
        return 'Binder'
    return row['Target_MoA_LLM']

pc_df['MoA'] = pc_df.apply(get_pc_moa, axis=1).apply(standardize_moa)
pc_df = pc_df[['Ligand_InChIKey', 'GPCR_UniProt', 'Assay_Layer', 'MoA', 'Source_DB']]

In [ ]:
# ==========================================
# 3. IUPHAR
# ==========================================
print("Processing IUPHAR...")
iuphar = pd.read_csv(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_MoA_Labeled.csv"))
iu_df = iuphar[['InChIKey', 'UniProt_AC', 'MoA_Label']].copy()

iu_df.rename(columns={'InChIKey': 'Ligand_InChIKey', 'UniProt_AC': 'GPCR_UniProt'}, inplace=True)
iu_df['Source_DB'] = 'IUPHAR'
iu_df['Assay_Layer'] = 'Unknown' # IUPHAR typically gives standard functional results without explicit cellular layer context
iu_df['MoA'] = iu_df['MoA_Label'].apply(standardize_moa)
iu_df = iu_df[['Ligand_InChIKey', 'GPCR_UniProt', 'Assay_Layer', 'MoA', 'Source_DB']]

In [ ]:
# ==========================================
# 4. GLASS
# ==========================================
print("Processing GLASS...")
glass = pd.read_csv(str(BASE / "Output/DB/GLASS/NAR/GLASS_v2_MoA_Labeled.csv"))
gl_df = glass[['compound_inchikey', 'UniProt_AC', 'MoA_Label']].copy()

gl_df.rename(columns={'compound_inchikey': 'Ligand_InChIKey', 'UniProt_AC': 'GPCR_UniProt'}, inplace=True)
gl_df['Source_DB'] = 'GLASS'
gl_df['Assay_Layer'] = 'Unknown' # GLASS uses text-mining from literature, layer is often ambiguous
gl_df['MoA'] = gl_df['MoA_Label'].apply(standardize_moa)
gl_df = gl_df[['Ligand_InChIKey', 'GPCR_UniProt', 'Assay_Layer', 'MoA', 'Source_DB']]

In [ ]:
# ==========================================
# 5. Merge, Clean, and Export
# ==========================================
print("Merging datasets...")
master_df = pd.concat([cb_df, pc_df, iu_df, gl_df], ignore_index=True)

# Drop any rows missing crucial keys and remove exact duplicates
master_df.dropna(subset=['Ligand_InChIKey', 'GPCR_UniProt'], inplace=True)
master_df.drop_duplicates(inplace=True)

# Export the Final Master Table
output_path = str(BASE / "Output/DB/GPCRactDB/Bioassay_MoA_Master_Integrated.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
master_df.to_csv(output_path, index=False)

print(f"\nSuccessfully saved the Master Table to: {output_path}")
print(f"Total Unique Assay Records: {len(master_df):,}")
print("\n[Assay Layer Distribution]")
print(master_df['Assay_Layer'].value_counts())
print("\n[MoA Distribution]")
print(master_df['MoA'].value_counts())

#### DrugDB (Human Efficacy) MoA Master tb

In [ ]:
import pandas as pd
import numpy as np
import os

def standardize_moa(moa):
    """표준 6-Class MoA로 매핑합니다."""
    if pd.isna(moa): return 'Unknown'
    moa = str(moa).strip().lower()
    
    if moa in ['full agonist', 'agonist', 'activator', 'stimulator']: return 'Agonist'
    elif moa in ['antagonist', 'inhibitor', 'blocker']: return 'Antagonist'
    elif moa in ['partial agonist']: return 'Partial Agonist'
    elif moa in ['inverse agonist']: return 'Inverse Agonist'
    elif moa in ['pam', 'positive allosteric modulator']: return 'PAM'
    elif moa in ['nam', 'negative allosteric modulator']: return 'NAM'
    elif moa in ['binder', 'modulator', 'ligand', 'other']: return 'Binder'
    
    return 'Unknown'

def load_and_standardize_drug_db(filepath, inchi_col, uniprot_col, moa_col, source_name, tier):
    """각 DB의 포맷을 표준화하여 데이터프레임으로 반환합니다."""
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found.")
        return pd.DataFrame()
        
    df = pd.read_csv(filepath)
    df = df[[inchi_col, uniprot_col, moa_col]].copy()
    df.rename(columns={inchi_col: 'Ligand_InChIKey', uniprot_col: 'GPCR_UniProt', moa_col: 'Raw_MoA'}, inplace=True)
    
    df['Human_MoA'] = df['Raw_MoA'].apply(standardize_moa)
    df['Source_DB'] = source_name
    df['Tier'] = tier
    
    # 결측치 및 Unknown/Binder 제거 (인체 '효능' 정답지이므로 기능성 라벨만 취합)
    df.dropna(subset=['Ligand_InChIKey', 'GPCR_UniProt'], inplace=True)
    df = df[~df['Human_MoA'].isin(['Unknown', 'Binder'])]
    
    return df[['Ligand_InChIKey', 'GPCR_UniProt', 'Human_MoA', 'Source_DB', 'Tier']]

In [ ]:
# ==========================================
# 1. 6개 데이터베이스 개별 로드 및 포맷팅
# ==========================================
print("Loading and standardizing Human Efficacy databases...")

# Tier 1 (Gold Standard: 임상 검증 약물 DB)
db_drugbank = load_and_standardize_drug_db(str(BASE / "Output/DB/DrugBank/NAR/DrugBank_MoA_Labeled.csv"), 'InChIKey', 'Target_UNIPROT', 'MoA_Label', 'DrugBank', 'Tier 1')
db_drugcentral = load_and_standardize_drug_db(str(BASE / "Output/DB/DrugCentral/NAR/DrugCentral_MoA_Labeled.csv"), 'inchikey', 'ACCESSION', 'MoA_Label', 'DrugCentral', 'Tier 1')
db_dgidb = load_and_standardize_drug_db(str(BASE / "Output/DB/DGIdb/NAR/DGIdb_MoA_labeled.csv"), 'Ikey', 'AC', 'Label', 'DGIdb', 'Tier 1')
db_gpcrdb = load_and_standardize_drug_db(str(BASE / "DB/GPCRdb//GPCRdb_gpcr_moa.csv"), 'INCHIKey', 'Uniprot', 'ModeOfAction', 'GPCRdb', 'Tier 1')

# Tier 1.5 (Therapeutic 편향 및 미승인 표준약리물질 다수 포함)
db_ttd = load_and_standardize_drug_db(str(BASE / "Output/DB/TTD/NAR/TTD_MoA_Labeled.csv"), 'InChIKey', 'UniProt_AC', 'MoA_Label', 'TTD', 'Tier 1.5')
db_iuphar = load_and_standardize_drug_db(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_MoA_Labeled.csv"), 'InChIKey', 'UniProt_AC', 'MoA_Label', 'IUPHAR', 'Tier 1.5')

In [ ]:
# ==========================================
# 2. 통합 및 충돌(Conflict) 해결 로직
# ==========================================
all_drug_moa = pd.concat([db_drugbank, db_drugcentral, db_dgidb, db_gpcrdb, db_ttd, db_iuphar], ignore_index=True)
all_drug_moa.drop_duplicates(inplace=True)

def resolve_moa_conflicts(group):
    """
    동일 약물-타겟 쌍에 대해 DB간 라벨이 충돌할 경우의 병합 기준
    1) Tier 1이 존재하면 Tier 1.5는 무시
    2) Agonist vs Antagonist 정면 충돌 시 'Conflict' 라벨 부여 (노이즈 파기용)
    3) Partial / Full Agonist 혼재 시 더 포괄적인 대표 라벨로 병합 등
    """
    sources = ", ".join(group['Source_DB'].unique())
    tiers = group['Tier'].unique()
    
    # Tier 1이 하나라도 있으면 Tier 1.5 배제
    if 'Tier 1' in tiers:
        valid_moas = group[group['Tier'] == 'Tier 1']['Human_MoA'].unique()
        final_tier = 'Tier 1'
    else:
        valid_moas = group['Human_MoA'].unique()
        final_tier = 'Tier 1.5'
        
    valid_moas = set(valid_moas)
    
    # 만장일치인 경우
    if len(valid_moas) == 1:
        return pd.Series({'Human_MoA': valid_moas.pop(), 'Source_DBs': sources, 'Final_Tier': final_tier})
        
    # Agonist 계열과 Antagonist 계열 정면 충돌 파기
    agonist_family = {'Agonist', 'Partial Agonist'}
    antagonist_family = {'Antagonist', 'Inverse Agonist'}
    
    if (valid_moas & agonist_family) and (valid_moas & antagonist_family):
        return pd.Series({'Human_MoA': 'Conflict_Drop', 'Source_DBs': sources, 'Final_Tier': final_tier})
        
    # 같은 계열 내 충돌 병합 (예: Agonist + Partial Agonist -> Agonist)
    if valid_moas.issubset(agonist_family):
        return pd.Series({'Human_MoA': 'Agonist', 'Source_DBs': sources, 'Final_Tier': final_tier})
    if valid_moas.issubset(antagonist_family):
        return pd.Series({'Human_MoA': 'Antagonist', 'Source_DBs': sources, 'Final_Tier': final_tier})
        
    # 그 외 (PAM/NAM 등과 충돌 시 보수적으로 Conflict 처리)
    return pd.Series({'Human_MoA': 'Conflict_Drop', 'Source_DBs': sources, 'Final_Tier': final_tier})

print("Resolving inter-database conflicts...")
master_human_moa = all_drug_moa.groupby(['Ligand_InChIKey', 'GPCR_UniProt']).apply(resolve_moa_conflicts).reset_index()

# 충돌로 인해 파기된 데이터(Conflict_Drop) 필터링
conflict_count = len(master_human_moa[master_human_moa['Human_MoA'] == 'Conflict_Drop'])
master_human_moa = master_human_moa[master_human_moa['Human_MoA'] != 'Conflict_Drop']

In [ ]:
# ==========================================
# 3. Export Final Tier 1/1.5 Ground Truth Table
# ==========================================
output_path = str(BASE / "Output/DB/GPCRactDB/Human_MoA_Master_Integrated.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
master_human_moa.to_csv(output_path, index=False)

print(f"\nSuccessfully built Human MoA Master Table: {output_path}")
print(f"- Total Unique Verified Human Efficacy Pairs: {len(master_human_moa):,}")
print(f"- Unresolvable Conflicts Dropped (e.g. Agonist vs Antagonist): {conflict_count:,}")
print("\n[Human MoA Distribution]")
print(master_human_moa['Human_MoA'].value_counts())
print("\n[Tier Distribution]")
print(master_human_moa['Final_Tier'].value_counts())